In [65]:
import pandas as pd   
import numpy as np    
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import seaborn as sns
import matplotlib.pyplot as plt


### 데이터 불러오기

In [66]:
df = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("submission_example.csv")

### 데이터 설명

| 변수명           | 데이터타입 | 설명                        | 범주 값 또는 예시                                                                                                                         |
| ------------- | ----- | ------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **age**       | 수치형   | 나이                        | -                                                                                                                                  |
| **job**       | 범주형   | 직업                        | admin, unknown, unemployed, management, housemaid, entrepreneur, student, blue-collar, self-employed, retired, technician, service |
| **marital**   | 범주형   | 결혼상태                      | married, divorced,single (사별해도 divorced로 포함)                                                                                                          |
| **education** | 범주형   | 학력                        | unknown, secondary, primary, tertiary                                                                                              |
| **default**   | 범주형   | 신용 채무 불이행 여부              | yes, no                                                                                                                            |
| **balance**   | 수치형   | 연평균 잔액 (단위: 유로)           | -                                                                                                                                  |
| **housing**   | 범주형   | 주택 대출 여부                  | yes, no                                                                                                                            |
| **loan**      | 범주형   | 개인 대출 여부                  | yes, no                                                                                                                            |
| **contact**   | 범주형   | 최근 연락 수단                  | unknown, telephone, cellular                                                                                                       |
| **day**       | 수치형   | 최근 연락 날짜                  | 1 \~ 31                                                                                                                            |
| **month**     | 범주형   | 최근 연락한 달                  | jan, feb, mar, ..., dec                                                                                                            |
| **duration**  | 수치형   | 최근 통화 시간 (초)              | -                                                                                                                                  |
| **campaign**  | 수치형   | 이번 캠페인 중 고객에게 연락한 횟수      | -                                                                                                                                  |
| **pdays**     | 수치형   | 이전 캠페인 이후 경과일 (-1은 연락 없음) | -                                                                                                                                  |
| **previous**  | 수치형   | 이전 캠페인에서 고객에게 연락한 횟수      | -                                                                                                                                  |
| **poutcome**  | 범주형   | 이전 캠페인 결과                 | unknown, other, failure, success                                                                                                   |
| **label**     | 범주형   | 정기예금 가입 여부 (타겟 변수)        | 1 = 가입, 0 = 미가입                                                                                                                    |


## EDA

### 결측치 확인

#### Target

- 범주형 비율 해석

    - job
        - student, retired, unemployed : 양수 -> 가입자 비중이 더 높음
        - blue-collar, services : 음수 -> 마케팅 효율이 낮은 직업에 해당

    - education
        - tertiary  > 양수에 해당하므로 고학력자일 수록 정기예금에 관심이 많다는 것을 해석할 수 있음
    
    - contact
        - cellular : 휴대폰에 해당하는데 이로 연락을 했을 때 가입할 확률이 올라감
    
    - month
        - mar apr, oct, sep : 양수 
        - may : significantly : 음수

    - martial
        - single : 양수 
            -> 독신일 경우 독립적이기에 가입가능성이 올라가는 것으로 해석 가능
        
        - married : 음수 

    - poutcome 
        - success : 이전 캠페인시 성공시 가입확률 올라감
        - unknown : 과거 기록이 없을 때는 음수일 경우가 많기에 주의

    - default, loan, housing
        - 신용 불이행 X, 대출이 없을 수록 가입확률이 높아짐


## 수치형 변수

### train_test data 나누기

In [94]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['label' , 'ID'])  
y = df['label']               
num_col = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
cat_col = ['month','job','education','poutcome', 'default','loan', 'housing','marital','contact','day']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## PipeLine

#### 범주형 변수 pipeline 작성

- one hot encoder : binary 변수 일때 사용
    - 'default', 'loan', 'housing' (Y/N) , 'marital', 'contact' (유의미한 most_frequent에 해당하는 범주가 있음)
- target encder : 범주와 타겟간의 연관성이 명확할 때 사용
    - job, poutcome,
- ordinal encoder : 순서 있는 범주에 사용
    - education

In [95]:
#!pip install category_encoders 

In [96]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np

# 월 -> 숫자 변환
month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
             'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}

education_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['primary', 'secondary', 'tertiary']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

binary_col = ['default', 'loan', 'housing']
binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore'))
])

other_col = ['marital', 'contact']
other_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

def add_paseng(df):
    df = df.copy()
    df['month_num'] = df['month'].map(month_map).astype(int)
    # 시간성 제고
    df['day_of_year'] = df['day'] + (df['month_num'] - 1) * 30
    # poutcome_success
    df['poutcome_success'] = (df['poutcome'] == 'success').astype(int)
    return df[['day_of_year', 'poutcome_success']]

day_poutcome_transformer = Pipeline([
    ('adder', FunctionTransformer(add_paseng, validate=False))
])

job_edu_transformer = Pipeline([
    ('concat', FunctionTransformer(
        lambda df: (df['job'] + '_' + df['education']).to_frame(name='job_education'),
        validate=False
    )),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

log_transformer = FunctionTransformer(np.log1p, validate=True)


In [97]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.compose import ColumnTransformer

#pdays 에 대해서는 -1 (연락없음)과 같은 특수 카테고리기 때문에 -1 변수를 999로 할당함.
pdays_transformer = Pipeline([
    ('minus1_to_nan', FunctionTransformer(lambda x: x.replace(-1, np.nan))),
    ('imputer', SimpleImputer(strategy='constant', fill_value=999)),
])

num_preprocessor = ColumnTransformer(transformers=[
    ('age', StandardScaler(), ['age']),
    ('balance', PowerTransformer(method='yeo-johnson'), ['balance']),
    ('duration', PowerTransformer(method='yeo-johnson'), ['duration']),
    ('campaign', RobustScaler(), ['campaign']),
    ('previous', PowerTransformer(method='yeo-johnson'), ['previous']),
    ('pdays', pdays_transformer, ['pdays'])
], remainder='drop')

num_pipeline_no_pt = ColumnTransformer([
    ('age',      StandardScaler(), ['age']),
    ('balance',  RobustScaler(),   ['balance']),
    ('duration', RobustScaler(),   ['duration']),
    ('campaign', RobustScaler(),   ['campaign']),
    ('previous','passthrough',     ['previous']),
    ('pdays',    pdays_transformer,['pdays']),
])

num_pipeline_log = ColumnTransformer([
    ('age',      StandardScaler(), ['age']),
    ('balance',  Pipeline([
        ('log', log_transformer),
    ]), ['balance']),
    ('duration', Pipeline([
        ('log', log_transformer),
    ]), ['duration']),
    ('campaign', RobustScaler(),   ['campaign']),
    ('previous','passthrough',     ['previous']),
    ('pdays',    pdays_transformer,['pdays']),
])


In [98]:
cat_preprocessor = ColumnTransformer(
    transformers=[
        ('job_edu',   job_edu_transformer,     ['job', 'education']),
        ('education', education_transformer,   ['education']),
        ('binary',    binary_transformer,      binary_col),
        ('other',     other_transformer,       other_col),
        ('day_pout',  day_poutcome_transformer,['day', 'month', 'poutcome'])
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

In [99]:
num_pipeline_log.set_output(transform='pandas')



ColumnTransformer(transformers=[('age', StandardScaler(), ['age']),
                                ('balance',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>,
                                                                      validate=True))]),
                                 ['balance']),
                                ('duration',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>,
                                                                      validate=True))]),
                                 ['duration']),
                                ('campaign', RobustScaler(), ['campaign']),
                                ('previous', 'passthrough', ['previous']),
                                ('pdays',
                                 Pipeline(steps=[('minus1_to_nan',
                                                  FunctionTransformer(func=<function <lambda> at 0x3094e8ae0>)),
                                                 ('imputer',
                                                  SimpleImputer(fill_value=999,
                                                                strategy='constant'))]),
                                 ['pdays'])])

In [100]:

cat_preprocessor.set_output(transform='pandas')

ColumnTransformer(transformers=[('job_edu',
                                 Pipeline(steps=[('concat',
                                                  FunctionTransformer(func=<function <lambda> at 0x3094e8720>)),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['job', 'education']),
                                ('education',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['primary',
                                                                              's...
                                 ['default', 'loan', 'housing']),
                                ('other',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['marital', 'contact']),
                                ('day_pout',
                                 Pipeline(steps=[('adder',
                                                  FunctionTransformer(func=<function add_paseng at 0x3094e82c0>))]),
                                 ['day', 'month', 'poutcome'])],
                  verbose_feature_names_out=False)

#### 모델 num_preprocessor

In [101]:
model_pipeline_num1 = ColumnTransformer(transformers=[
    ('cat', cat_preprocessor, cat_col),
    ('num', num_preprocessor, num_col) 
])

model_pipeline_num1.set_output(transform='pandas')

ColumnTransformer(transformers=[('cat',
                                 ColumnTransformer(transformers=[('job_edu',
                                                                  Pipeline(steps=[('concat',
                                                                                   FunctionTransformer(func=<function <lambda> at 0x3094e8720>)),
                                                                                  ('ohe',
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['job',
                                                                   'education']),
                                                                 ('education',
                                                                  Pipeline(steps=[('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('ordinal',
                                                                                   O...
                                                                  PowerTransformer(),
                                                                  ['duration']),
                                                                 ('campaign',
                                                                  RobustScaler(),
                                                                  ['campaign']),
                                                                 ('previous',
                                                                  PowerTransformer(),
                                                                  ['previous']),
                                                                 ('pdays',
                                                                  Pipeline(steps=[('minus1_to_nan',
                                                                                   FunctionTransformer(func=<function <lambda> at 0x3094e8ae0>)),
                                                                                  ('imputer',
                                                                                   SimpleImputer(fill_value=999,
                                                                                                 strategy='constant'))]),
                                                                  ['pdays'])]),
                                 ['age', 'balance', 'duration', 'campaign',
                                  'pdays', 'previous'])])

In [102]:
model_pipeline_num1.fit(X_train, y_train) #target_encoding 때문에 y_train 필요로 함
X_train_pipe1 = model_pipeline_num1.transform(X_train)
X_val_pipe1 = model_pipeline_num1.transform(X_val)

#### 모델 num_pipeline_no_pt

In [103]:
model_pipeline_no_pt = ColumnTransformer(transformers=[
    ('cat', cat_preprocessor, cat_col),
    ('num', num_pipeline_no_pt, num_col) 
])

model_pipeline_no_pt.set_output(transform='pandas')

ColumnTransformer(transformers=[('cat',
                                 ColumnTransformer(transformers=[('job_edu',
                                                                  Pipeline(steps=[('concat',
                                                                                   FunctionTransformer(func=<function <lambda> at 0x3094e8720>)),
                                                                                  ('ohe',
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['job',
                                                                   'education']),
                                                                 ('education',
                                                                  Pipeline(steps=[('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('ordinal',
                                                                                   O...
                                                                 ('duration',
                                                                  RobustScaler(),
                                                                  ['duration']),
                                                                 ('campaign',
                                                                  RobustScaler(),
                                                                  ['campaign']),
                                                                 ('previous',
                                                                  'passthrough',
                                                                  ['previous']),
                                                                 ('pdays',
                                                                  Pipeline(steps=[('minus1_to_nan',
                                                                                   FunctionTransformer(func=<function <lambda> at 0x3094e8ae0>)),
                                                                                  ('imputer',
                                                                                   SimpleImputer(fill_value=999,
                                                                                                 strategy='constant'))]),
                                                                  ['pdays'])]),
                                 ['age', 'balance', 'duration', 'campaign',
                                  'pdays', 'previous'])])

In [104]:
model_pipeline_no_pt.fit(X_train, y_train) #target_encoding 때문에 y_train 필요로 함
X_train_pipe2 = model_pipeline_no_pt.transform(X_train)
X_val_pipe2 = model_pipeline_no_pt.transform(X_val)

#### pt 적용 여부에 따라 f1 스코어의 변화를 봄

In [105]:
from lightgbm import LGBMClassifier

In [106]:
# 1) PowerTransformer 적용 버전
pipe_with_pt = Pipeline([
    ('pre', model_pipeline_num1),   
    ('clf', LGBMClassifier(random_state=42))
])

# 2) PowerTransformer 미적용 버전
pipe_no_pt = Pipeline([
    ('pre', model_pipeline_no_pt),     
    ('clf', LGBMClassifier(random_state=42))
])

pipe_with_pt.fit(X_train, y_train)
pipe_no_pt.fit(X_train, y_train)


preds1 = pipe_with_pt.predict(X_val)
preds2 = pipe_no_pt.predict(X_val)



In [107]:
print("With PowerTransformer    F1-score:", f1_score(y_val, preds1))
print("Without PowerTransformer F1-score:", f1_score(y_val, preds2))

With PowerTransformer    F1-score: 0.5362872421695951
Without PowerTransformer F1-score: 0.5348659003831417


#### 로그변환!

In [108]:
log_preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_log, num_col),
        ('cat', cat_preprocessor,   cat_col)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

        

model_pipeline_log = Pipeline(steps=[
    ('preproc', log_preprocessor),
    ('clf',     LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'))
])


In [109]:
log_preprocessor.set_output(transform='pandas')

ColumnTransformer(transformers=[('num',
                                 ColumnTransformer(transformers=[('age',
                                                                  StandardScaler(),
                                                                  ['age']),
                                                                 ('balance',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['balance']),
                                                                 ('duration',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['duration']),
                                                                 ('campaign',
                                                                  RobustScaler(),
                                                                  ['campa...
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['marital',
                                                                   'contact']),
                                                                 ('day_pout',
                                                                  Pipeline(steps=[('adder',
                                                                                   FunctionTransformer(func=<function add_paseng at 0x3094e82c0>))]),
                                                                  ['day',
                                                                   'month',
                                                                   'poutcome'])],
                                                   verbose_feature_names_out=False),
                                 ['month', 'job', 'education', 'poutcome',
                                  'default', 'loan', 'housing', 'marital',
                                  'contact', 'day'])],
                  verbose_feature_names_out=False)

In [110]:
model_pipeline_log.named_steps['preproc'].set_output(transform='pandas')


ColumnTransformer(transformers=[('num',
                                 ColumnTransformer(transformers=[('age',
                                                                  StandardScaler(),
                                                                  ['age']),
                                                                 ('balance',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['balance']),
                                                                 ('duration',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['duration']),
                                                                 ('campaign',
                                                                  RobustScaler(),
                                                                  ['campa...
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['marital',
                                                                   'contact']),
                                                                 ('day_pout',
                                                                  Pipeline(steps=[('adder',
                                                                                   FunctionTransformer(func=<function add_paseng at 0x3094e82c0>))]),
                                                                  ['day',
                                                                   'month',
                                                                   'poutcome'])],
                                                   verbose_feature_names_out=False),
                                 ['month', 'job', 'education', 'poutcome',
                                  'default', 'loan', 'housing', 'marital',
                                  'contact', 'day'])],
                  verbose_feature_names_out=False)

In [111]:
model_pipeline_log.fit(X_train,y_train)
preds_log = model_pipeline_log.predict(X_val)


print("Log-transform F1:", f1_score(y_val, preds_log))

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

Log-transform F1: 0.5958041958041959


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

In [112]:
print("Log-transform F1:", f1_score(y_val, preds_log))

Log-transform F1: 0.5958041958041959


In [113]:
#Log-transform F1: 0.5426829268292683 >> 로그 스케일링이 훨씬 좋음~
#Log-transform F1: 0.5958041958041959 >> 로그 스케일링 + verbose =-1 + class_weight : balanced

In [114]:
import pandas as pd

# 1) fit + transform (DataFrame)
X_trn_df = log_preprocessor.fit_transform(X_train, y_train)
# 2) transform 만 (DataFrame)
X_val_df = log_preprocessor.transform(X_val)

# 컬럼명 확인: 문자열로 잘 나오는지
print(X_trn_df.columns[:10])


Index([                               0,                                1,
                                      2,                                3,
                                      4,                                5,
         'job_education_admin._primary', 'job_education_admin._secondary',
        'job_education_admin._tertiary',   'job_education_admin._unknown'],
      dtype='object')


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

In [115]:
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector

In [116]:
base_model = LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')

In [117]:
sfs = SequentialFeatureSelector(
    estimator=base_model,
    n_features_to_select=15,   # 예: 최종 10개 선택
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)


In [118]:
sfs.fit(X_trn_pre, y_train)

TypeError: Feature names are only supported if all input features have string names, but your input has ['int', 'str'] as feature name / column name types. If you want feature names to be stored and validated, you must convert them all to strings, by using X.columns = X.columns.astype(str) for example. Otherwise you can remove feature / column names from your input data, or convert them all to a non-string data type.

In [119]:
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector

In [ ]:
base_model = LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')

In [ ]:
sfs = SequentialFeatureSelector(
    estimator=base_model,
    n_features_to_select=15,   # 예: 최종 10개 선택
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)


In [ ]:
sfs.fit(X_trn_pre, y_train)

TypeError: Feature names are only supported if all input features have string names, but your input has ['int', 'str'] as feature name / column name types. If you want feature names to be stored and validated, you must convert them all to strings, by using X.columns = X.columns.astype(str) for example. Otherwise you can remove feature / column names from your input data, or convert them all to a non-string data type.

In [ ]:
total_cols = [x for x in X_train.columns]

id_cols = ['ID']
y_cols = ['label']
x_cols = [x for x in total_cols if x not in id_cols+y_cols]

In [ ]:
# feature와 target분리
df_trn_x = X_trn_num[x_cols]
df_trn_y = X_val_num[y_cols].values.ravel()
df_tst_x = test[x_cols]

KeyError: "['age', 'job', 'marital', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome'] not in index"

In [ ]:
# 검증 데이터셋 분리
df_retrn_x, df_val_x, df_retrn_y, df_val_y = train_test_split(df_trn_x, df_trn_y, test_size=0.2, random_state=42, stratify=df_trn_y)

In [ ]:
# 초기 모델 정의
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
#학습
base_model.fit(df_retrn_x, df_retrn_y)

LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

sfs = SequentialFeatureSelector(
    estimator = pipeline_log,
    n_features_to_select=10,     
    direction='forward',         
    scoring='f1',                
    cv=5,                        
    n_jobs=-1                    
)

In [ ]:
sfs.fit(X_train, y_train)

ValueError: could not convert string to float: 'management'

In [ ]:
selected = list(np.array(preprocessor.get_feature_names_out())[sfs.get_support()])
print("선택된 변수들:", selected)


NameError: name 'preprocessor' is not defined

In [ ]:
# 5) 선택된 변수로 재학습 & 최종 성능 검증
X_train_sel = sfs.transform(X_train)
X_val_sel   = sfs.transform(X_val)

model.fit(X_train_sel, y_train)
preds = model.predict(X_val_sel)

from sklearn.metrics import f1_score
print("Forward Selection F1-score:", f1_score(y_val, preds))

## Modeling

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(probability=True),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42)
}

### Optuna

In [ ]:
# optimizing 10 번 진행

- RandomForest, XGBoost, LogisticRegression 하이퍼파라미터 튜닝
- StackingClassifier로 세 모델 앙상블 구성
- predict_proba()로 확률값 추출
- Threshold를 0.3 ~ 0.7 사이에서 변화시켜 F1-score를 최적화
- 최적 threshold에서의 F1-score를 Optuna의 objective값으로 반환

In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import f1_score

In [ ]:
def objective(trial):
    rf = RandomForestClassifier(
        n_estimators=trial.suggest_int("rf_n_estimators", 50, 200),
        max_depth=trial.suggest_int("rf_max_depth", 3, 15),
        class_weight="balanced",
        random_state=42
    )
    xgb = XGBClassifier(
        n_estimators=trial.suggest_int("xgb_n_estimators", 50, 200),
        max_depth=trial.suggest_int("xgb_max_depth", 3, 15),
        learning_rate=trial.suggest_float("xgb_lr", 0.01, 0.3),
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=trial.suggest_float("xgb_spw", 1.0, 5.0),
    )
    lr = LogisticRegression(
        C=trial.suggest_float("lr_C", 0.01, 10.0),
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )

    clf = StackingClassifier(
        estimators=[("rf", rf), ("xgb", xgb), ("lr", lr)],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5,
        passthrough=True  # 원본 특성도 최종 모델에 전달
    )

    clf.fit(X_train_pipe, y_train)
    probs = clf.predict_proba(X_val_pipe)[:, 1]

    # Threshold tuning
    thresholds = np.linspace(0.3, 0.7, 21)
    f1_scores = [f1_score(y_val, (probs >= t).astype(int)) for t in thresholds]
    
    return max(f1_scores)

import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best F1-score:", study.best_value)
print("Best params:", study.best_params)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-05-29 13:22:21,177] A new study created in memory with name: no-name-65193996-ccc9-412c-b181-745f6b34b8e6
[W 2025-05-29 13:22:21,179] Trial 0 failed with parameters: {'rf_n_estimators': 165, 'rf_max_depth': 8, 'xgb_n_estimators': 195, 'xgb_max_depth': 13, 'xgb_lr': 0.26763226565640924, 'xgb_spw': 4.341479329066593, 'lr_C': 6.882093726424097} because of the following error: NameError("name 'X_train_pipe' is not defined").
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_57098/3279250901.p

NameError: name 'X_train_pipe' is not defined

In [ ]:
best_params = study.best_params

final_rf = RandomForestClassifier(
    n_estimators=best_params["rf_n_estimators"],
    max_depth=best_params["rf_max_depth"],
    class_weight="balanced",
    random_state=42
)
final_xgb = XGBClassifier(
    n_estimators=best_params["xgb_n_estimators"],
    max_depth=best_params["xgb_max_depth"],
    learning_rate=best_params["xgb_lr"],
    scale_pos_weight=best_params["xgb_spw"],
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
final_lr = LogisticRegression(
    C=best_params["lr_C"],
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

final_stack = StackingClassifier(
    estimators=[("rf", final_rf), ("xgb", final_xgb), ("lr", final_lr)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    passthrough=True
)

final_stack.fit(X_train_pipe, y_train)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [18:45:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [18:45:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [18:45:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [18:45:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encode

StackingClassifier(cv=5,
                   estimators=[('rf',
                                RandomForestClassifier(class_weight='balanced',
                                                       max_depth=3,
                                                       n_estimators=142,
                                                       random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='loglo...
                                              max_delta_step=None, max_depth=3,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=96, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('lr',
                                LogisticRegression(C=0.13109887684072988,
                                                   class_weight='balanced',
                                                   max_iter=1000,
                                                   random_state=42))],
                   final_estimator=LogisticRegression(max_iter=1000),
                   passthrough=True)

In [ ]:
# Threshold 다시 튜닝해서 적용
final_probs = final_stack.predict_proba(X_val_pipe)[:, 1]
thresholds = np.linspace(0.3, 0.7, 21)
f1s = [f1_score(y_val, (final_probs >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1s)]
print(f"Best threshold: {best_threshold:.2f}, F1: {np.max(f1s):.4f}")

Best threshold: 0.30, F1: 0.6005


### 최종 test 데이터에 적용

In [ ]:
test_pipe = model_pipeline.transform(test)

test_probs = final_stack.predict_proba(test_pipe)[:, 1]

test_pred = (test_probs >= best_threshold).astype(int)


/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_68829/3913398071.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(month_map)


## Submission

In [ ]:
submission = pd.DataFrame({
    "ID": test["ID"],        # ID 컬럼이 있다면
    "label": test_pred
})
submission.to_csv("submission_try01.csv", index=False)